# Day 065 | Activation Functions Deep Dive

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** May 16, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Activation function | Non-linear transform applied after each layer | Converts raw neuron output into a useful signal |
| Sigmoid / Tanh | Saturating functions — crush output to a fixed range | Historical defaults; now rarely used in hidden layers |
| ReLU | max(0, z) — zero for negatives, identity for positives | Default choice for hidden layers |
| Leaky ReLU / ELU | Fixes dying ReLU by allowing small negative gradient | Useful when neurons keep dying during training |
| GELU | Smooth probabilistic gating — used in Transformers | State-of-the-art for NLP and attention models |
| Vanishing gradient | Gradients shrink to near zero through many sigmoid layers | Training stalls; early layers fail to learn |
| Output activation | Depends on task: linear, sigmoid, or softmax | Regression vs binary vs multi-class classification |

---

## Our Geotechnical Problem

We are building a neural network to predict **shear wave velocity (Vs)** from SPT N60, depth, and unit weight. This is a **regression task** — the output is a continuous positive number (m/s).

The choice of **activation function** in hidden layers dramatically affects:
- Whether gradients vanish during backpropagation
- How fast the network converges
- Whether neurons "die" (always output zero)
- Final prediction accuracy

Today we dissect every major activation function and select the right one for our Vs regression network.

---

## Part 1: Setup and Activation Function Formulas

Each activation function applies an element-wise transformation to the pre-activation `z = W·x + b`:

| Function | Formula | Output Range | Key Property |
|----------|---------|-------------|-------------|
| Sigmoid | σ(z) = 1/(1+e⁻ᶻ) | (0, 1) | Saturates at both ends |
| Tanh | (eᶻ − e⁻ᶻ)/(eᶻ + e⁻ᶻ) | (−1, 1) | Zero-centered, still saturates |
| ReLU | max(0, z) | [0, ∞) | Simple, fast, sparse activations |
| Leaky ReLU | max(αz, z), α≪1 | (−∞, ∞) | Allows small negative gradient |
| ELU | z if z>0, α(eᶻ−1) otherwise | (−α, ∞) | Smooth negative, near-zero mean |
| GELU | z · Φ(z) | (≈−0.17, ∞) | Probabilistic smooth gating |

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print("Libraries loaded.")
print(f"PyTorch version: {torch.__version__}")

---

## Part 2: Implement All Activations in NumPy — Function + Derivative Side by Side

In [ ]:
# ─────────────────────────────────────────────────────
# NUMPY IMPLEMENTATIONS — each activation and its derivative
# ─────────────────────────────────────────────────────

z = np.linspace(-4, 4, 400)

def sigmoid(z):         return 1 / (1 + np.exp(-z))
def d_sigmoid(z):       s = sigmoid(z); return s * (1 - s)

def tanh_fn(z):         return np.tanh(z)
def d_tanh(z):          return 1 - np.tanh(z)**2

def relu(z):            return np.maximum(0, z)
def d_relu(z):          return (z > 0).astype(float)

def leaky_relu(z, a=0.01): return np.where(z > 0, z, a * z)
def d_leaky_relu(z, a=0.01): return np.where(z > 0, 1.0, a)

def elu(z, a=1.0):      return np.where(z > 0, z, a * (np.exp(z) - 1))
def d_elu(z, a=1.0):    return np.where(z > 0, 1.0, a * np.exp(z))

def gelu(z):
    """GELU approximation: z * 0.5 * (1 + tanh(sqrt(2/pi)*(z + 0.044715*z^3)))."""
    return z * 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

def d_gelu(z):
    eps = 1e-5
    return (gelu(z + eps) - gelu(z - eps)) / (2 * eps)

activations = [
    ("Sigmoid",    sigmoid,    d_sigmoid,    "tab:blue"),
    ("Tanh",       tanh_fn,    d_tanh,       "tab:orange"),
    ("ReLU",       relu,       d_relu,       "tab:green"),
    ("Leaky ReLU", leaky_relu, d_leaky_relu, "tab:red"),
    ("ELU",        elu,        d_elu,        "tab:purple"),
    ("GELU",       gelu,       d_gelu,       "tab:brown"),
]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Activation Functions — f(z) (top) and f'(z) Derivative (bottom)",
             fontsize=13, fontweight='bold')

for col, (name, fn, dfn, color) in enumerate(activations):
    ax0, ax1 = axes[0, col], axes[1, col]

    ax0.plot(z, fn(z), color=color, lw=2)
    ax0.axhline(0, color='gray', lw=0.5, ls='--')
    ax0.axvline(0, color='gray', lw=0.5, ls='--')
    ax0.set_title(name, fontsize=10, fontweight='bold')
    ax0.set_ylim(-1.6, 1.6)
    ax0.set_xlabel("z")
    if col == 0: ax0.set_ylabel("f(z)")

    ax1.plot(z, dfn(z), color=color, lw=2, ls='--')
    ax1.axhline(0, color='gray', lw=0.5, ls='--')
    ax1.set_ylim(-0.4, 1.4)
    ax1.set_xlabel("z")
    if col == 0: ax1.set_ylabel("f'(z)")

plt.tight_layout()
plt.show()

print("Key: Sigmoid derivative peaks at 0.25. Tanh peaks at 1.0.")
print("ReLU derivative is 1 for z>0 and 0 elsewhere — clean gradient flow.")

---

## Part 3: Vanishing Gradient Demo — Sigmoid vs ReLU Through 10 Layers

Backpropagation multiplies gradients at every layer. Sigmoid's derivative is at most **0.25**, so after 10 layers the gradient is at most 0.25¹⁰ ≈ 10⁻⁷ — vanished. ReLU's derivative is 1 for positive inputs, so gradients remain healthy.

In [ ]:
# ─────────────────────────────────────────────────────
# VANISHING GRADIENT — track gradient magnitude through layers
# ─────────────────────────────────────────────────────

np.random.seed(0)
n_layers = 10
x_init = np.random.randn(100)


def gradient_flow(x_init, fn, dfn, n_layers=10):
    """
    Simulate mean gradient magnitude layer by layer.
    At each layer, multiply the running gradient by the local derivative.
    """
    x = x_init.copy()
    cumulative_grad = np.ones_like(x)
    magnitudes = []
    for _ in range(n_layers):
        W = np.random.randn(*x.shape) * 0.5
        z = W * x
        x = fn(z)
        cumulative_grad *= dfn(z)
        magnitudes.append(np.mean(np.abs(cumulative_grad)))
    return magnitudes


sig_g   = gradient_flow(x_init, sigmoid,    d_sigmoid)
tanh_g  = gradient_flow(x_init, tanh_fn,    d_tanh)
relu_g  = gradient_flow(x_init, relu,       d_relu)
lrelu_g = gradient_flow(x_init, leaky_relu, d_leaky_relu)

layers = np.arange(1, n_layers + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for grads, label, color in [
    (sig_g,   'Sigmoid',    'tab:blue'),
    (tanh_g,  'Tanh',       'tab:orange'),
    (relu_g,  'ReLU',       'tab:green'),
    (lrelu_g, 'Leaky ReLU', 'tab:red'),
]:
    ax1.plot(layers, grads, color=color, lw=2, marker='o', ms=5, label=label)
    ax2.semilogy(layers, [max(g, 1e-15) for g in grads],
                 color=color, lw=2, marker='o', ms=5, label=label)

for ax in (ax1, ax2):
    ax.set_xlabel("Layer depth")
    ax.legend()
    ax.grid(True, alpha=0.3)

ax1.set_ylabel("Mean |gradient|"); ax1.set_title("Linear scale")
ax2.set_ylabel("Mean |gradient| (log)"); ax2.set_title("Log scale — vanishing is clear!")
plt.suptitle("Vanishing Gradient: 10 layers deep", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"After {n_layers} layers:")
print(f"  Sigmoid:    {sig_g[-1]:.2e}")
print(f"  Tanh:       {tanh_g[-1]:.2e}")
print(f"  ReLU:       {relu_g[-1]:.4f}")
print(f"  Leaky ReLU: {lrelu_g[-1]:.4f}")
print("\nConclusion: Sigmoid gradients vanish to ~0. ReLU gradients stay healthy.")

---

## Part 4: The Dying ReLU Problem and How Leaky ReLU Fixes It

If a neuron's pre-activation `z` is always negative, ReLU outputs **exactly zero** and its gradient is also zero. The neuron is permanently dead — no gradient flows through it, so it never recovers no matter how long you train.

In [ ]:
# ─────────────────────────────────────────────────────
# DYING ReLU — show dead neurons vs Leaky ReLU / ELU
# ─────────────────────────────────────────────────────

np.random.seed(7)
# Simulate pre-activations shifted very negative (as if a big LR update pushed weights down)
z_vals = np.random.randn(300) - 2.5

relu_out  = relu(z_vals)
lrelu_out = leaky_relu(z_vals, a=0.1)
elu_out   = elu(z_vals, a=1.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
configs_dying = [
    (relu_out,  'ReLU',               'tab:green'),
    (lrelu_out, 'Leaky ReLU (α=0.1)', 'tab:red'),
    (elu_out,   'ELU (α=1.0)',         'tab:purple'),
]

for ax, (out, name, color) in zip(axes, configs_dying):
    dead = int(np.sum(out == 0))
    ax.scatter(range(len(out)), out, c=color, s=6, alpha=0.6)
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_title(f"{name}\n{dead}/{len(out)} neurons dead (output=0)", fontsize=10)
    ax.set_xlabel("Neuron index"); ax.set_ylabel("Activation output")
    ax.grid(True, alpha=0.2)

plt.suptitle("Dying ReLU Problem: neurons stuck at zero vs alternatives",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

dead_relu  = int(np.sum(relu_out == 0))
dead_lrelu = int(np.sum(lrelu_out == 0))
print(f"Pre-activation mean: {z_vals.mean():.2f}  (mostly negative)")
print(f"ReLU:       {dead_relu}/{len(z_vals)} neurons dead ({100*dead_relu/len(z_vals):.0f}%)")
print(f"Leaky ReLU: {dead_lrelu}/{len(z_vals)} neurons dead")
print()
print("Remedies for dying ReLU:")
print("  1. Leaky ReLU (alpha=0.01-0.1) — small gradient keeps neuron alive")
print("  2. ELU — smooth exponential tail, zero-mean activations")
print("  3. He (Kaiming) initialization — prevents weights starting too negative")
print("  4. Lower learning rate — prevents large weight updates that kill neurons")

---

## Part 5: Which Activation for Which Layer?

| Layer Position | Recommended | Reason | Geotechnical Example |
|---------------|------------|--------|---------------------|
| Hidden layers (default) | **ReLU** | Fast, no vanishing gradient, sparse | Vs regression hidden layers |
| Hidden layers (deep) | **Leaky ReLU / ELU** | Avoids dead neurons | 10+ layer settlement networks |
| Hidden layers (state-of-the-art) | **GELU** | Smooth, used in Transformers | Foundation sequence models |
| Output — regression | **Linear (none)** | Unbounded real output | Vs (m/s), settlement (mm) |
| Output — binary class | **Sigmoid** | Probability in (0, 1) | Liquefaction risk |
| Output — multi-class | **Softmax** | Probability distribution | USCS soil classification |
| Recurrent cells | **Tanh** | Bounded, zero-centered | Soil time-series RNN |

In [ ]:
# ─────────────────────────────────────────────────────
# OUTPUT LAYER ACTIVATIONS — demonstration
# ─────────────────────────────────────────────────────

print("=" * 55)
print("OUTPUT ACTIVATION EXAMPLES")
print("=" * 55)

# 1. Regression: no activation (linear)
raw_regression = np.array([150.0, 305.0, 450.0, 90.0])  # Vs raw outputs
print("\n1. Regression (Vs in m/s) — Linear output (no activation):")
print(f"   Raw network output: {raw_regression}")
print(f"   Final prediction:   {raw_regression}  <-- unchanged")

# 2. Binary classification: sigmoid
raw_binary = np.array([-2.1, -0.4, 0.8, 2.7])  # liquefaction logits
probs_binary = sigmoid(raw_binary)
print("\n2. Binary (liquefaction risk) — Sigmoid output:")
print(f"   Raw logits:   {raw_binary}")
print(f"   Probabilities: {np.round(probs_binary, 3)}")
print(f"   Predictions:   {['No risk' if p < 0.5 else 'Risk' for p in probs_binary]}")

# 3. Multi-class: softmax
def softmax(z): e = np.exp(z - z.max()); return e / e.sum()
raw_multi = np.array([3.1, 1.0, 0.2])  # logits for Sand, Clay, Gravel
probs_multi = softmax(raw_multi)
classes = ['Sand', 'Clay', 'Gravel']
print("\n3. Multi-class (soil type) — Softmax output:")
print(f"   Raw logits:    {raw_multi}")
print(f"   Probabilities: {np.round(probs_multi, 3)}  (sum={probs_multi.sum():.1f})")
print(f"   Prediction:    {classes[np.argmax(probs_multi)]}")

---

## Part 6: Build 3 PyTorch Networks for Vs Prediction — Compare Training Loss

In [ ]:
# ─────────────────────────────────────────────────────
# SYNTHETIC DATA: Vs from N60, depth, unit weight
# ─────────────────────────────────────────────────────

np.random.seed(42)
torch.manual_seed(42)

n = 600
N60     = np.random.uniform(5, 60, n)
depth   = np.random.uniform(1, 30, n)
gamma   = np.random.uniform(15, 21, n)

# Imai & Tonouchi (1982): Vs = 97 * N^0.314, plus depth and unit weight effects
Vs = (97 * N60**0.314
      + 3.0 * depth
      + 5.0 * (gamma - 16)
      + np.random.randn(n) * 15)

X = np.column_stack([N60, depth, gamma]).astype(np.float32)
y = Vs.astype(np.float32)

X_mu, X_sd = X.mean(0), X.std(0)
y_mu, y_sd = y.mean(), y.std()
Xn = (X - X_mu) / X_sd
yn = (y - y_mu) / y_sd

Xt = torch.tensor(Xn)
yt = torch.tensor(yn).unsqueeze(1)

split = int(0.8 * n)
X_tr, X_va = Xt[:split], Xt[split:]
y_tr, y_va = yt[:split], yt[split:]

print(f"Dataset: {n} samples | {split} train | {n - split} val")
print(f"Vs range: {Vs.min():.0f} — {Vs.max():.0f} m/s  (mean {Vs.mean():.0f} m/s)")

In [ ]:
# ─────────────────────────────────────────────────────
# THREE NETWORKS — same architecture, different hidden activation
# ─────────────────────────────────────────────────────

class VsNet(nn.Module):
    """3-input → 64 → 64 → 32 → 1 regression network."""
    def __init__(self, act):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), act,
            nn.Linear(64, 64), act,
            nn.Linear(64, 32), act,
            nn.Linear(32, 1),         # linear output for regression
        )
    def forward(self, x): return self.net(x)


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=300, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    tr_losses, va_losses = [], []
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = criterion(model(X_tr), y_tr)
        loss.backward()
        opt.step()
        tr_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            va_losses.append(criterion(model(X_va), y_va).item())
    return tr_losses, va_losses


configs_pt = [
    ("Sigmoid",    nn.Sigmoid(),       'tab:blue'),
    ("ReLU",       nn.ReLU(),          'tab:green'),
    ("Leaky ReLU", nn.LeakyReLU(0.1), 'tab:red'),
]

torch.manual_seed(42)
results = {}
for name, act, color in configs_pt:
    model = VsNet(act)
    tr_l, va_l = train_model(model, X_tr, y_tr, X_va, y_va, epochs=300)
    results[name] = {'train': tr_l, 'val': va_l, 'color': color, 'model': model}
    print(f"{name:14s} | final train loss: {tr_l[-1]:.4f} | val loss: {va_l[-1]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT — training vs validation loss curves
# ─────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for name, d in results.items():
    ax1.plot(d['train'], color=d['color'], lw=2, label=name)
    ax2.plot(d['val'],   color=d['color'], lw=2, label=name)

ax1.set_title("Training Loss (MSE)"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("MSE")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_title("Validation Loss (MSE)"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("MSE")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle("Vs Prediction: Activation Function Comparison (same architecture)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation:")
print("  Sigmoid: slow convergence due to vanishing gradients in early layers.")
print("  ReLU / Leaky ReLU: converge significantly faster and to lower final loss.")
print("  For Vs regression: ReLU or Leaky ReLU is the practical choice.")

---

## Part 7: Practical Guidance Table

| Activation | Use When | Avoid When |
|-----------|----------|----------|
| **ReLU** | Default hidden layer, most tasks | Very deep networks with dying neurons |
| **Leaky ReLU** | Dead neurons appear during training | Simplicity is the priority |
| **ELU** | Need smooth outputs, zero-mean activations | Computationally tight deployment |
| **GELU** | Transformers, large modern networks | Small shallow networks (overkill) |
| **Sigmoid** | Binary output layer only | Any hidden layer (vanishing gradient) |
| **Tanh** | LSTM/GRU recurrent cells | Deep feedforward hidden layers |
| **Linear** | Regression output (Vs, settlement) | Hidden layers (removes non-linearity) |
| **Softmax** | Multi-class output (soil type) | Hidden layers |

**Quick rule for geotechnical ML:**
- Regression (Vs, settlement, bearing capacity) → **ReLU hidden + linear output**
- Binary (liquefaction, failure) → **ReLU hidden + sigmoid output**
- Multi-class (soil type, zone) → **ReLU hidden + softmax output**

---

## Practice Exercises

### Exercise 1: Add GELU to the comparison
Extend the three-network experiment to include `nn.GELU()`. Compare final validation loss.

### Exercise 2: Sweep Leaky ReLU alpha
Train Leaky ReLU networks with alpha ∈ {0.001, 0.01, 0.1, 0.3}. Plot final validation loss vs alpha.

### Exercise 3: Gradient norm per layer
Build a 15-layer Sigmoid network and a 15-layer ReLU network. After one backward pass, print the gradient norm at each layer and plot them side by side.

In [ ]:
# Exercise 1: Add GELU
# configs_ext = configs_pt + [("GELU", nn.GELU(), 'tab:brown')]
# torch.manual_seed(42)
# for name, act, color in configs_ext:
#     model = VsNet(act)
#     tr_l, va_l = train_model(model, X_tr, y_tr, X_va, y_va, epochs=300)
#     print(f"{name:14s} | val loss: {va_l[-1]:.4f}")

# Exercise 2: Sweep alpha
# alphas = [0.001, 0.01, 0.1, 0.3]
# final_val = []
# for a in alphas:
#     model = VsNet(nn.LeakyReLU(a))
#     _, va_l = train_model(model, X_tr, y_tr, X_va, y_va, epochs=300)
#     final_val.append(va_l[-1])
# plt.plot(alphas, final_val, 'o-'); plt.xlabel('alpha'); plt.ylabel('Val loss')
# plt.title('Leaky ReLU alpha sweep'); plt.show()

---

## Key Takeaways

1. **Sigmoid and Tanh saturate** — derivatives reach at most 0.25 and 1.0, causing gradients to vanish exponentially in deep networks.

2. **ReLU is the practical default** for hidden layers: gradient is exactly 1 for positive inputs, enabling healthy backpropagation through many layers.

3. **Dying ReLU**: if a neuron's pre-activation is always negative, it permanently outputs 0 with zero gradient — it can never recover. Fix with Leaky ReLU, ELU, or careful initialization.

4. **Leaky ReLU** allows α·z for negative z (typically α=0.01), preventing neuron death while keeping most of ReLU's benefits.

5. **GELU** is the modern state-of-the-art for large models (BERT, GPT, ViT) — smooth probabilistic gating.

6. **Output activation is task-specific**: linear for regression, sigmoid for binary, softmax for multi-class.

7. **For Vs regression in geotechnical ML**: ReLU or Leaky ReLU hidden layers + linear output is the practical winning combination.

---

## How This Connects to ML

| Activation Concept | Broader ML Equivalent |
|--------------------|----------------------|
| ReLU in hidden layers | Standard in ResNet, VGG, MobileNet |
| GELU | BERT, GPT-3/4, Vision Transformers |
| Sigmoid output | Logistic regression — a 1-layer sigmoid network |
| Softmax output | Every image classifier final layer |
| Tanh in cells | LSTM forget / input / output gates |
| Vanishing gradient | Why ResNets use skip connections |
| Dying ReLU | Why BatchNorm before ReLU is common practice |

---

*Day 065 of 100 — The right activation turns raw numbers into learning signals!*